In [ ]:
import pandas as pd
import numpy as np


RAW_DATA_PATH = "employee_data.csv"          # <-- point this at your actual CSV
OUTPUT_PATH = "employee_engagement_processed.csv"

COL = {
    "age": "Age",
    "attrition": "Attrition",              # 0/1 OR "Yes"/"No" — handled below
    "business_travel": "BusinessTravel",
    "department": "Department",
    "distance": "DistanceFromHome",
    "education": "Education",
    "education_field": "EducationField",
    "env_satisfaction": "EnvironmentSatisfaction",
    "gender": "Gender",
    "job_involvement": "JobInvolvement",
    "job_level": "JobLevel",
    "job_role": "JobRole",
    "job_satisfaction": "JobSatisfaction",
    "marital_status": "MaritalStatus",
    "monthly_income": "MonthlyIncome",
    "num_companies": "NumCompaniesWorked",
    "overtime": "OverTime",                # "Yes"/"No"
    "salary_hike": "PercentSalaryHike",
    "performance_rating": "PerformanceRating",
    "relationship_satisfaction": "RelationshipSatisfaction",
    "stock_option": "StockOptionLevel",
    "total_working_years": "TotalWorkingYears",
    "training_times": "TrainingTimesLastYear",
    "worklife_balance": "WorkLifeBalance",
    "years_at_company": "YearsAtCompany",
    "years_in_role": "YearsInCurrentRole",
    "years_since_promotion": "YearsSinceLastPromotion",
    "years_with_manager": "YearsWithCurrManager",
}

# 1-4 ordinal satisfaction/engagement fields that feed the Engagement Index
ENGAGEMENT_FIELDS = ["job_involvement", "job_satisfaction",
                     "env_satisfaction", "relationship_satisfaction"]


def load_data(path=RAW_DATA_PATH):
    df = pd.read_csv(path)
    print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
    return df

Data validation and normalization


In [ ]:
def phase1_validate_and_normalize(df):
    df = df.copy()

    # Normalize Attrition to 0/1 regardless of source encoding (numeric or Yes/No)
    attr_raw = df[COL["attrition"]].astype(str).str.strip().str.lower()
    df[COL["attrition"]] = attr_raw.replace(
        {"yes": "1", "no": "0"}
    ).astype(int)

    # Validate ordinal 1-4 scales; clip out-of-range and flag missing
    ordinal_cols = [COL[f] for f in ENGAGEMENT_FIELDS] + [COL["worklife_balance"]]
    for c in ordinal_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        n_missing = df[c].isna().sum()
        if n_missing:
            # Impute missing ordinal ratings with the column median (robust to skew)
            median_val = df[c].median()
            df[c] = df[c].fillna(median_val)
            print(f"  {c}: imputed {n_missing} missing values with median ({median_val})")
        out_of_range = ~df[c].between(1, 4)
        if out_of_range.any():
            df.loc[out_of_range, c] = df.loc[out_of_range, c].clip(1, 4)
            print(f"  {c}: clipped {out_of_range.sum()} out-of-range values to [1,4]")

    # Normalize each ordinal field to 0-1 for comparability across scales
    for c in ordinal_cols:
        df[c + "_norm"] = (df[c] - 1) / 3  # maps 1-4 -> 0-1

    return df

Engagement Index Construction


In [ ]:
def phase2_engagement_index(df):
    df = df.copy()
    norm_cols = [COL[f] + "_norm" for f in ENGAGEMENT_FIELDS]

    # Composite engagement score: equal-weighted average of the 4 normalized dimensions
    df["EngagementIndex"] = df[norm_cols].mean(axis=1) * 100  # scale to 0-100

    # Satisfaction Stability Score: how consistent the 4 dimensions are for a person
    # (low std dev = stable/consistent satisfaction; high std dev = volatile)
    df["SatisfactionStabilityScore"] = 100 - (df[norm_cols].std(axis=1) * 100)

    return df

Burnout Risk Identification

In [ ]:
def phase3_burnout_risk(df):
    df = df.copy()
    ot_col, wlb_col = COL["overtime"], COL["worklife_balance"]

    df["IsOvertime"] = df[ot_col].astype(str).str.strip().str.lower().eq("yes")
    df["LowWorkLifeBalance"] = df[wlb_col] <= 2

    def classify(row):
        if row["IsOvertime"] and row["LowWorkLifeBalance"]:
            return "High"
        elif row["IsOvertime"] or row["LowWorkLifeBalance"]:
            return "Medium"
        return "Low"

    df["BurnoutRiskLevel"] = df.apply(classify, axis=1)

    # Numeric Burnout Risk Score (0-100): weights overtime + inverse work-life balance
    df["BurnoutRiskScore"] = (
        df["IsOvertime"].astype(int) * 50
        + (4 - df[wlb_col]) / 3 * 50
    )

    return df


Workload & Stress Analysis

In [ ]:
def phase4_workload_stress(df):
    df = df.copy()
    travel_col, dist_col = COL["business_travel"], COL["distance"]

    # Workload Stress Indicator: combines travel intensity + overtime
    travel_weight = df[travel_col].map({
        "Non-Travel": 0, "Travel_Rarely": 1, "Travel_Frequently": 2
    }).fillna(1)
    df["WorkloadStressIndicator"] = (
        travel_weight * 25 + df["IsOvertime"].astype(int) * 50
    ).clip(0, 100)

    # Commute bucket for long vs short commute comparisons
    df["CommuteBucket"] = pd.cut(
        df[dist_col], bins=[-1, 5, 15, 100],
        labels=["Short (<=5)", "Medium (6-15)", "Long (16+)"]
    )

    return df



Career-Stage Engagement Analysis

In [ ]:
def phase5_career_stage(df):
    df = df.copy()
    tenure_col, role_years_col = COL["years_at_company"], COL["years_in_role"]

    df["TenureBucket"] = pd.cut(
        df[tenure_col], bins=[-1, 2, 5, 10, 100],
        labels=["0-2 yrs", "3-5 yrs", "6-10 yrs", "10+ yrs"]
    )

    # Stagnation flag: long time in company/role but no promotion recently
    promo_col = COL["years_since_promotion"]
    df["StagnationFlag"] = (
        (df[tenure_col] >= 4) & (df[promo_col] >= 3)
    )

    return df

Engagement vs Attrition

In [ ]:
def phase6_attrition_context(df):
    # This phase doesn't add columns; it's a comparison run at reporting time.
    # Kept here for structure/documentation — see summarize_attrition_context().
    return df


def summarize_attrition_context(df):
    summary = df.groupby(COL["attrition"])[
        ["EngagementIndex", "BurnoutRiskScore", "WorkloadStressIndicator"]
    ].mean().rename(index={0: "Stayed", 1: "Left"})
    return summary.round(2)



KPI ROLL-UP


In [ ]:
def compute_kpis(df):
    kpis = {
        "Engagement Index (avg)": round(df["EngagementIndex"].mean(), 2),
        "Burnout Risk Score (avg)": round(df["BurnoutRiskScore"].mean(), 2),
        "% High Burnout Risk": round((df["BurnoutRiskLevel"] == "High").mean() * 100, 2),
        "Work-Life Balance Index (avg)": round(df[COL["worklife_balance"]].mean(), 2),
        "Satisfaction Stability Score (avg)": round(df["SatisfactionStabilityScore"].mean(), 2),
        "Workload Stress Indicator (avg)": round(df["WorkloadStressIndicator"].mean(), 2),
        "Attrition Rate %": round(df[COL["attrition"]].mean() * 100, 2),
    }
    return kpis


Main

In [ ]:
def run_pipeline(path=RAW_DATA_PATH, output_path=OUTPUT_PATH):
    df = load_data(path)

    print("\nPhase 1: Data Validation & Normalization")
    df = phase1_validate_and_normalize(df)

    print("\nPhase 2: Engagement Index Construction")
    df = phase2_engagement_index(df)

    print("\nPhase 3: Burnout Risk Identification")
    df = phase3_burnout_risk(df)

    print("\nPhase 4: Workload & Stress Analysis")
    df = phase4_workload_stress(df)

    print("\nPhase 5: Career-Stage Engagement Analysis")
    df = phase5_career_stage(df)

    print("\nPhase 6: Engagement vs Attrition (contextual)")
    df = phase6_attrition_context(df)

    print("\n--- KPIs ---")
    for k, v in compute_kpis(df).items():
        print(f"  {k}: {v}")

    print("\n--- Engagement vs Attrition summary ---")
    print(summarize_attrition_context(df))

    df.to_csv(output_path, index=False)
    print(f"\nSaved processed dataset -> {output_path}")
    return df


if __name__ == "__main__":
    run_pipeline()

FileNotFoundError: [Errno 2] No such file or directory: 'employee_data.csv'

In [ ]:
def run_pipeline(path=RAW_DATA_PATH, output_path=OUTPUT_PATH):
    df = load_data(path)

    print("\nPhase 1: Data Validation & Normalization")
    df = phase1_validate_and_normalize(df)

    print("\nPhase 2: Engagement Index Construction")
    df = phase2_engagement_index(df)

    print("\nPhase 3: Burnout Risk Identification")
    df = phase3_burnout_risk(df)

    print("\nPhase 4: Workload & Stress Analysis")
    df = phase4_workload_stress(df)

    print("\nPhase 5: Career-Stage Engagement Analysis")
    df = phase5_career_stage(df)

    print("\nPhase 6: Engagement vs Attrition (contextual)")
    df = phase6_attrition_context(df)

    print("\n--- KPIs ---")
    for k, v in compute_kpis(df).items():
        print(f"  {k}: {v}")

    print("\n--- Engagement vs Attrition summary ---")
    print(summarize_attrition_context(df))

    df.to_csv(output_path, index=False)
    print(f"\nSaved processed dataset -> {output_path}")
    return df


if __name__ == "__main__":
    run_pipeline()
    import os
print(os.getcwd())
print(os.listdir())

Loaded 1470 rows, 31 columns

Phase 1: Data Validation & Normalization

Phase 2: Engagement Index Construction

Phase 3: Burnout Risk Identification

Phase 4: Workload & Stress Analysis

Phase 5: Career-Stage Engagement Analysis

Phase 6: Engagement vs Attrition (contextual)

--- KPIs ---
  Engagement Index (avg): 57.44
  Burnout Risk Score (avg): 34.8
  % High Burnout Risk: 8.57
  Work-Life Balance Index (avg): 2.76
  Satisfaction Stability Score (avg): 68.39
  Workload Stress Indicator (avg): 41.31
  Attrition Rate %: 16.12

--- Engagement vs Attrition summary ---
           EngagementIndex  BurnoutRiskScore  WorkloadStressIndicator
Attrition                                                            
Stayed               58.79             32.04                    38.14
Left                 50.42             49.16                    57.81

Saved processed dataset -> employee_engagement_processed.csv
/content
['.config', 'employee_engagement_processed.csv', 'employee_data.csv', 'sampl

In [ ]:
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'sample_data']


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving employee_data.csv to employee_data.csv


In [ ]:
import os
print(os.listdir())

['.config', 'sample_data']


In [ ]:
import os
print(os.listdir())


In [1]:
!pip install streamlit -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 23.9 MB/s eta 0:00:00


In [2]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸

In [3]:
import os

os.makedirs("palo-alto-hr-dashboard/data", exist_ok=True)
print("Folders created")

Folders created


In [4]:
import shutil

shutil.copy("employee_data.csv", "palo-alto-hr-dashboard/data/employee_data.csv")
print("Dataset copied")

FileNotFoundError: [Errno 2] No such file or directory: 'employee_data.csv'

In [5]:
!ls

palo-alto-hr-dashboard	sample_data


In [6]:
from google.colab import files
uploaded = files.upload()

Saving employee_data.csv to employee_data.csv


In [7]:
!ls

employee_data.csv  palo-alto-hr-dashboard  sample_data


In [8]:
import shutil
shutil.copy("employee_data.csv", "palo-alto-hr-dashboard/data/employee_data.csv")
print("Dataset copied")

Dataset copied


In [9]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
import os
project_path = "/content/drive/MyDrive/palo-alto-hr-dashboard"
os.makedirs(project_path + "/data", exist_ok=True)
print("Created at:", project_path)

Created at: /content/drive/MyDrive/palo-alto-hr-dashboard


In [12]:
from google.colab import files
uploaded = files.upload()

Saving employee_data.csv to employee_data (1).csv


In [13]:
import shutil
shutil.copy("employee_data.csv", project_path + "/data/employee_data.csv")
print("Dataset saved permanently to Drive")

Dataset saved permanently to Drive


In [14]:
df = pd.read_csv("/content/drive/MyDrive/palo-alto-hr-dashboard/data/employee_data.csv")

NameError: name 'pd' is not defined

In [15]:
import pandas as pd

In [16]:
df = pd.read_csv("/content/drive/MyDrive/palo-alto-hr-dashboard/data/employee_data.csv")
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,3,1,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,4,4,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,3,2,0,7,3,3,0,0,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,...,3,3,0,8,3,3,8,7,3,0
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,...,3,4,1,6,3,3,2,2,2,2


In [18]:
df = pd.read_csv("/content/drive/MyDrive/palo-alto-hr-dashboard/data/employee_data.csv")

In [19]:
"data_path": "/content/drive/MyDrive/palo-alto-hr-dashboard/data/employee_data.csv",

SyntaxError: illegal target for annotation (2910977042.py, line 1)

In [20]:
CONFIG = {
    "age_col": "Age",
    "attrition_col": "Attrition",
    "department_col": "Department",
    "overtime_col": "OverTime",
    "worklife_col": "WorkLifeBalance",
    "job_involvement_col": "JobInvolvement",
    "job_satisfaction_col": "JobSatisfaction",
    "env_satisfaction_col": "EnvironmentSatisfaction",
    "relationship_satisfaction_col": "RelationshipSatisfaction",
    "job_role_col": "JobRole",
    "job_level_col": "JobLevel",
    "years_at_company_col": "YearsAtCompany",
    "years_current_role_col": "YearsInCurrentRole",
    "business_travel_col": "BusinessTravel",
    "distance_col": "DistanceFromHome",
    "data_path": "/content/drive/MyDrive/palo-alto-hr-dashboard/data/employee_data.csv",
}

In [21]:
"data_path": "/content/drive/MyDrive/palo-alto-hr-dashboard/data/employee_data.csv",

SyntaxError: illegal target for annotation (2910977042.py, line 1)

In [22]:
df = pd.read_csv(CONFIG["data_path"])
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,3,1,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,4,4,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,3,2,0,7,3,3,0,0,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,...,3,3,0,8,3,3,8,7,3,0
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,...,3,4,1,6,3,3,2,2,2,2


In [23]:
%%writefile /content/drive/MyDrive/palo-alto-hr-dashboard/app.py
import streamlit as st
import pandas as pd
import plotly.express as px

# ============================
# CONFIG
# ============================
CONFIG = {
    "age_col": "Age",
    "attrition_col": "Attrition",
    "department_col": "Department",
    "overtime_col": "OverTime",
    "worklife_col": "WorkLifeBalance",
    "job_involvement_col": "JobInvolvement",
    "job_satisfaction_col": "JobSatisfaction",
    "env_satisfaction_col": "EnvironmentSatisfaction",
    "relationship_satisfaction_col": "RelationshipSatisfaction",
    "job_role_col": "JobRole",
    "job_level_col": "JobLevel",
    "years_at_company_col": "YearsAtCompany",
    "years_current_role_col": "YearsInCurrentRole",
    "business_travel_col": "BusinessTravel",
    "distance_col": "DistanceFromHome",
    "data_path": "/content/drive/MyDrive/palo-alto-hr-dashboard/data/employee_data.csv",
}

st.set_page_config(page_title="PAN Engagement & Burnout Diagnostic", layout="wide")

@st.cache_data
def load_data(path):
    return pd.read_csv(path)

df = load_data(CONFIG["data_path"])

st.title("Employee Engagement, Satisfaction & Burnout Diagnostic")
st.caption("Palo Alto Networks — HR Analytics Case Study")

st.dataframe(df.head())

Writing /content/drive/MyDrive/palo-alto-hr-dashboard/app.py


In [24]:
!ls /content/drive/MyDrive/palo-alto-hr-dashboard/

app.py	data


In [25]:
!wget -q -O - https://loca.lt/mytunnelpassword

35.253.41.156

In [26]:
!wget -q -O - https://loca.lt/mytunnelpassword

35.253.41.156

In [27]:
!streamlit run /content/drive/MyDrive/palo-alto-hr-dashboard/app.py &>/content/logs.txt &

In [28]:
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦your url is: https://six-ads-relate.loca.lt
^C


In [29]:
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴your url is: https://nine-planes-chew.loca.lt
^C


In [32]:
!ls /content/drive/MyDrive/palo-alto-hr-dashboard/

app.py	data


In [33]:
!streamlit run /content/drive/MyDrive/palo-alto-hr-dashboard/app.py &>/content/logs.txt &

In [34]:
!cat /content/logs.txt



2026-08-13 06:50:20.335 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.253.41.156:8501



In [38]:
!wget -q -O - https://loca.lt/mytunnelpassword

^C


In [36]:
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴your url is: https://quiet-clowns-bet.loca.lt
^C


In [37]:
!cat /content/logs.txt



2026-08-13 06:50:20.335 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.253.41.156:8501

  Stopping...


In [39]:
!streamlit run /content/drive/MyDrive/palo-alto-hr-dashboard/app.py &>/content/logs.txt &

In [40]:
!cat /content/logs.txt



2026-08-13 07:00:00.150 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.253.41.156:8501



In [41]:
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴your url is: https://stupid-bags-lose.loca.lt
^C


In [42]:
!pip install pyngrok -q

In [43]:
from pyngrok import ngrok
public_url = ngrok.connect(8501)
print(public_url)

ERROR:pyngrok.process.ngrok:t=2026-08-13T07:24:17+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-08-13T07:24:17+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
CRITICAL:pyngrok.process.ngrok:t=2026-08-13T07:24:17+0000 lvl=crit msg="command failed" err="authentication failed: This ngrok session is not authenticated. ngrok requi

PyngrokNgrokError: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [2]:
from pyngrok import ngrok
ngrok.set_auth_token("3HqqU3Mea3HaDJdruosP4smudZc_4YvzDyAf9ctLPSfzp8WVr")

ModuleNotFoundError: No module named 'pyngrok'

In [3]:
!pip install streamlit pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 99.0 MB/s eta 0:00:00


In [4]:
import pandas as pd
import streamlit as st
from pyngrok import ngrok

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!ls /content/drive/MyDrive/palo-alto-hr-dashboard/

app.py	data


In [7]:
ngrok.set_auth_token("3HqqU3Mea3HaDJdruosP4smudZc_4YvzDyAf9ctLPSfzp8WVr")

In [8]:
!streamlit run /content/drive/MyDrive/palo-alto-hr-dashboard/app.py &>/content/logs.txt &

In [9]:
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://bloated-dress-dramatic.ngrok-free.dev" -> "http://localhost:8501"


In [10]:
%%writefile /content/drive/MyDrive/palo-alto-hr-dashboard/app.py
import streamlit as st
import pandas as pd
import plotly.express as px

# ============================
# CONFIG — edit only this block if column names change
# ============================
CONFIG = {
    "age_col": "Age",
    "attrition_col": "Attrition",
    "department_col": "Department",
    "overtime_col": "OverTime",
    "worklife_col": "WorkLifeBalance",
    "job_involvement_col": "JobInvolvement",
    "job_satisfaction_col": "JobSatisfaction",
    "env_satisfaction_col": "EnvironmentSatisfaction",
    "relationship_satisfaction_col": "RelationshipSatisfaction",
    "job_role_col": "JobRole",
    "job_level_col": "JobLevel",
    "years_at_company_col": "YearsAtCompany",
    "years_current_role_col": "YearsInCurrentRole",
    "business_travel_col": "BusinessTravel",
    "distance_col": "DistanceFromHome",
    "data_path": "/content/drive/MyDrive/palo-alto-hr-dashboard/data/employee_data.csv",
}

# ============================
# PAGE SETUP
# ============================
st.set_page_config(
    page_title="PAN Employee Engagement & Burnout Diagnostic",
    layout="wide",
    initial_sidebar_state="expanded"
)

@st.cache_data
def load_data(path):
    return pd.read_csv(path)

df = load_data(CONFIG["data_path"])

# ============================
# KPI CALCULATIONS
# ============================
df["EngagementIndex"] = df[[
    CONFIG["job_involvement_col"],
    CONFIG["job_satisfaction_col"],
    CONFIG["env_satisfaction_col"],
    CONFIG["relationship_satisfaction_col"]
]].mean(axis=1)

def compute_burnout_risk(row):
    if row[CONFIG["overtime_col"]] == "Yes" and row[CONFIG["worklife_col"]] <= 2:
        return "High"
    elif row[CONFIG["overtime_col"]] == "Yes" or row[CONFIG["worklife_col"]] <= 2:
        return "Medium"
    else:
        return "Low"

df["BurnoutRisk"] = df.apply(compute_burnout_risk, axis=1)

# ============================
# TITLE
# ============================
st.title("Employee Engagement, Satisfaction & Burnout Diagnostic")
st.caption("Palo Alto Networks — HR Analytics Case Study")

# ============================
# SIDEBAR FILTERS
# ============================
st.sidebar.header("Filters")

dept_filter = st.sidebar.multiselect(
    "Department",
    options=sorted(df[CONFIG["department_col"]].unique()),
    default=sorted(df[CONFIG["department_col"]].unique())
)

overtime_filter = st.sidebar.selectbox("Overtime", options=["All", "Yes", "No"])

engagement_threshold = st.sidebar.slider(
    "Minimum Engagement Index",
    min_value=float(df["EngagementIndex"].min()),
    max_value=float(df["EngagementIndex"].max()),
    value=float(df["EngagementIndex"].min())
)

tenure_range = st.sidebar.slider(
    "Years at Company",
    min_value=int(df[CONFIG["years_at_company_col"]].min()),
    max_value=int(df[CONFIG["years_at_company_col"]].max()),
    value=(0, int(df[CONFIG["years_at_company_col"]].max()))
)

filtered_df = df[df[CONFIG["department_col"]].isin(dept_filter)]
if overtime_filter != "All":
    filtered_df = filtered_df[filtered_df[CONFIG["overtime_col"]] == overtime_filter]
filtered_df = filtered_df[filtered_df["EngagementIndex"] >= engagement_threshold]
filtered_df = filtered_df[filtered_df[CONFIG["years_at_company_col"]].between(*tenure_range)]

st.sidebar.markdown(f"**Employees in view:** {len(filtered_df)}")

# ============================
# MODULE 1 — ENGAGEMENT HEALTH OVERVIEW
# ============================
st.header("1. Engagement Health Overview")

col1, col2, col3 = st.columns(3)
col1.metric("Org-Wide Engagement Score", round(filtered_df["EngagementIndex"].mean(), 2))
col2.metric("Employees in View", len(filtered_df))
col3.metric("High Burnout Risk %", f"{(filtered_df['BurnoutRisk'] == 'High').mean() * 100:.1f}%")

fig1 = px.histogram(filtered_df, x="EngagementIndex", nbins=20,
                     title="Engagement Score Distribution")
st.plotly_chart(fig1, use_container_width=True)

fig_sat = px.box(
    filtered_df.melt(
        value_vars=[
            CONFIG["job_involvement_col"], CONFIG["job_satisfaction_col"],
            CONFIG["env_satisfaction_col"], CONFIG["relationship_satisfaction_col"]
        ],
        var_name="Dimension", value_name="Score"
    ),
    x="Dimension", y="Score", title="Satisfaction Distribution by Dimension"
)
st.plotly_chart(fig_sat, use_container_width=True)

# ============================
# MODULE 2 — BURNOUT RISK DASHBOARD
# ============================
st.header("2. Burnout Risk Dashboard")

col4, col5 = st.columns(2)
with col4:
    fig2 = px.pie(filtered_df, names="BurnoutRisk", title="Burnout Risk Segments",
                  color="BurnoutRisk",
                  color_discrete_map={"High": "#d62728", "Medium": "#ff7f0e", "Low": "#2ca02c"})
    st.plotly_chart(fig2, use_container_width=True)

with col5:
    fig3 = px.box(filtered_df, x=CONFIG["overtime_col"], y="EngagementIndex",
                  color=CONFIG["overtime_col"], title="Engagement: Overtime vs Non-Overtime")
    st.plotly_chart(fig3, use_container_width=True)

fig_travel = px.bar(
    filtered_df.groupby(CONFIG["business_travel_col"])["EngagementIndex"].mean().reset_index(),
    x=CONFIG["business_travel_col"], y="EngagementIndex",
    title="Engagement by Travel Frequency"
)
st.plotly_chart(fig_travel, use_container_width=True)

# ============================
# MODULE 3 — ROLE & CAREER STAGE ANALYSIS
# ============================
st.header("3. Role & Career Stage Analysis")

fig4 = px.bar(
    filtered_df.groupby(CONFIG["job_role_col"])["EngagementIndex"].mean().sort_values().reset_index(),
    x="EngagementIndex", y=CONFIG["job_role_col"], orientation="h",
    title="Average Engagement by Job Role"
)
st.plotly_chart(fig4, use_container_width=True)

col6, col7 = st.columns(2)
with col6:
    fig5 = px.scatter(filtered_df, x=CONFIG["years_at_company_col"], y="EngagementIndex",
                       color="BurnoutRisk", title="Tenure vs Engagement")
    st.plotly_chart(fig5, use_container_width=True)

with col7:
    fig6 = px.box(filtered_df, x=CONFIG["job_level_col"], y="EngagementIndex",
                  title="Engagement by Job Level")
    st.plotly_chart(fig6, use_container_width=True)

# ============================
# MODULE 4 — MANAGER ACTION PANEL
# ============================
st.header("4. Manager Action Panel")

low_engagement = filtered_df[filtered_df["EngagementIndex"] < 2.5]
st.warning(f"⚠️ {len(low_engagement)} employees flagged with low engagement (Engagement Index < 2.5)")

high_risk = filtered_df[filtered_df["BurnoutRisk"] == "High"]
st.error(f"🔴 {len(high_risk)} employees flagged as High Burnout Risk")

st.subheader("Priority Intervention List")
st.dataframe(
    filtered_df[[
        CONFIG["job_role_col"], CONFIG["department_col"],
        "EngagementIndex", "BurnoutRisk", CONFIG["worklife_col"],
        CONFIG["overtime_col"], CONFIG["years_at_company_col"]
    ]].sort_values("EngagementIndex").reset_index(drop=True),
    use_container_width=True
)

Overwriting /content/drive/MyDrive/palo-alto-hr-dashboard/app.py


In [11]:
!pkill -f streamlit
!streamlit run /content/drive/MyDrive/palo-alto-hr-dashboard/app.py &>/content/logs.txt &

In [12]:
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://bloated-dress-dramatic.ngrok-free.dev" -> "http://localhost:8501"
